# Extension Deep Learning — CNN scratch vs pretrained

Ce notebook entraîne et évalue deux approches Deep Learning pour la détection de maladies sur feuilles :
32
1
- un petit CNN entraîné from-scratch,
- un modèle pretrained (ResNet18) fine-tuné (transfer learning).

L'objectif : comparer performances (accuracy / classification_report / matrice de confusion) et comparer aux modèles classiques basés sur features.


## Requirements

Installez les dépendances si nécessaire :

```bash
pip install torch torchvision timm scikit-learn matplotlib seaborn
```

Note: utilisez un GPU si disponible pour entraîner les modèles plus rapidement.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
# Enable cuDNN auto-tuner for fixed-size inputs to improve performance
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True
print('cudnn.benchmark:', torch.backends.cudnn.benchmark)
# DataLoader/transfer settings
pin_memory = True if torch.cuda.is_available() else False
use_amp = True if torch.cuda.is_available() else False


Device: cuda
cudnn.benchmark: True


In [2]:
# Dataset and transforms (adjust dataset_root if needed)
dataset_root = '../data/raw/PlantVillage'
img_size = 224
image_limit_per_class=50
train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2,0.2,0.2,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# Build image list from raw folders (same logic as the ML pipeline)
from PIL import Image
from sklearn.model_selection import train_test_split

class ImageListDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, labels, class_to_idx, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.class_to_idx = class_to_idx
        self.transform = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert('RGB')
        label = self.class_to_idx[self.labels[idx]]
        if self.transform is not None:
            img = self.transform(img)
        return img, label

class SubsetWithTransform(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = list(indices)
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        image, label = self.dataset[self.indices[idx]]
        if self.transform is not None:
            image = self.transform(image)
        return image, label

image_paths = []
labels_list = []
for class_name in os.listdir(dataset_root):
    class_path = os.path.join(dataset_root, class_name)
    if not os.path.isdir(class_path):
        continue
    files = [
        os.path.join(class_path, f)
        for f in os.listdir(class_path)
        if f.lower().endswith(('.jpg', '.png', '.jpeg'))
    ]
    # apply optional per-class limit if variable exists in the notebook
    try:
        limit = image_limit_per_class
    except NameError:
        limit = None
    if limit:
        files = files[:limit]
    for f in files:
        image_paths.append(f)
        labels_list.append(class_name)

if len(image_paths) == 0:
    raise FileNotFoundError(f'No images found under {dataset_root}')

class_names = sorted(list(set(labels_list)))
class_to_idx = {c:i for i,c in enumerate(class_names)}
base_dataset = ImageListDataset(image_paths, labels_list, class_to_idx, transform=None)
num_samples = len(base_dataset)
num_classes = len(class_names)
indices = np.arange(num_samples)
train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=labels_list, random_state=42)
train_dataset = SubsetWithTransform(base_dataset, train_idx, train_transform)
val_dataset = SubsetWithTransform(base_dataset, val_idx, val_transform)

batch_size = 32
num_workers = min(4, os.cpu_count() or 1)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
dataloaders = {'train': train_loader, 'val': val_loader}
dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}


In [3]:
### Skipping SimpleCNN (from-scratch)

#The from-scratch `SimpleCNN` is intentionally skipped in this notebook.
#We use the pretrained `ResNet18` (transfer learning) only for faster convergence and better accuracy.

# If you want to enable scratch training later, define and run a dedicated cell.


In [4]:
# Training / evaluation helpers (with optional AMP and non-blocking transfers)
from torch.cuda.amp import GradScaler, autocast

def train_epoch(model, loader, criterion, optimizer, device, scaler=None):
    model.train()
    running_loss = 0.0
    running_corrects = 0
    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        if scaler is not None:
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        preds = torch.argmax(outputs, dim=1)
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data).item()
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = running_corrects / len(loader.dataset)
    return epoch_loss, epoch_acc

def evaluate(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with autocast(enabled=(device.type == 'cuda')):
                outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    print('Accuracy:', acc)
    print(classification_report(all_labels, all_preds, target_names=class_names))
    return acc, all_labels, all_preds

def fit(model, dataloaders, criterion, optimizer, device, epochs=5, use_amp=True):
    history = {'train_loss': [], 'train_acc': []}
    scaler = GradScaler() if (use_amp and torch.cuda.is_available()) else None
    for epoch in range(epochs):
        loss, acc = train_epoch(model, dataloaders['train'], criterion, optimizer, device, scaler=scaler)
        history['train_loss'].append(loss)
        history['train_acc'].append(acc)
        print(f'Epoch {epoch+1}/{epochs} - loss: {loss:.4f} - acc: {acc:.4f}')
    return history


In [5]:
# SimpleCNN training skipped — using pretrained ResNet18 only
enable_scratch = False
print('SimpleCNN training skipped. Using ResNet18 (transfer learning) only.')


SimpleCNN training skipped. Using ResNet18 (transfer learning) only.


In [ ]:
# Pretrained (ResNet18) fine-tuning
try:
    from torchvision.models import resnet18, ResNet18_Weights
    weights = ResNet18_Weights.DEFAULT
    model_pre = resnet18(weights=weights)
except Exception:
    model_pre = models.resnet18(pretrained=True)

num_ftrs = model_pre.fc.in_features
model_pre.fc = nn.Linear(num_ftrs, num_classes)
model_pre = model_pre.to(device)

# Freeze backbone except final layer (fast experiment)
for name, param in model_pre.named_parameters():
    if 'fc' not in name:
        param.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model_pre.parameters()), lr=1e-4)
epochs = 3
history_pre = fit(model_pre, dataloaders, criterion, optimizer, device, epochs=epochs)
acc_pre, y_true_pre, y_pred_pre = evaluate(model_pre, dataloaders['val'], device)
torch.save(model_pre.state_dict(), '../results/combined_run/models/model_resnet18.pth')
print('Saved model_resnet18')


C:\Users\louay\AppData\Local\Temp\ipykernel_25932\2874865828.py:51: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if (use_amp and torch.cuda.is_available()) else None


In [ ]:
# Comparative evaluation vs classical features-based baseline (if features CSV exists)
results = []
results.append({'model':'ResNet18_transfer','accuracy':float(acc_pre)})
features_csv = '../results/combined_run/features/features.csv'
if os.path.exists(features_csv):
    df = pd.read_csv(features_csv).dropna()
    X_cols = [c for c in df.columns if c not in ('image','label')]
    X = df[X_cols].values
    y = df['label'].values
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestClassifier
    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, stratify=y_enc, random_state=42)
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    acc_rf = accuracy_score(y_test, y_pred)
    results.append({'model':'RF_features','accuracy':float(acc_rf)})
    print('RandomForest (features) accuracy:', acc_rf)
else:
    print('features CSV not found — skipping classical baseline')

results_df = pd.DataFrame(results).sort_values(by='accuracy', ascending=False)
print(results_df)

plt.figure(figsize=(6,4))
sns.barplot(x='model', y='accuracy', data=results_df)
plt.ylim(0,1)
plt.title('Model comparison (accuracy)')
plt.show()


## Conclusion & next steps

- Exécutez ce notebook sur GPU pour de meilleures performances et augmentez `epochs` pour une évaluation sérieuse.
- Comparez : accuracy, classification_report, matrice de confusion, runtime, et taille du modèle.
- Options avancées : cross-validation, augmentation plus forte, scheduler LR, fine-tuning plus profond, tests ensemblistes.

Souhaitez-vous que j'intègre ces cellules dans le notebook principal ou que j'exécute un court entraînement de test (si vous autorisez l'exécution ici) ?